# 2/2 - Benchmark a checkpoint

Scores a trained checkpoint on the full PU-Net grid and prints the row into
the published comparison table.

## Before running
1. *Add Input* -> `pointdenoise-code`
2. *Add Input* -> `pointdenoise-data`
3. *Add Input* -> the dataset holding your `best.pt`
4. GPU on

Roughly an hour. Calibration runs first and asserts, because a harness that
does not reproduce a published number produces results comparable to nothing.


In [ ]:
import glob, os, subprocess, sys

def show_tree(root="/kaggle/input", limit=40):
    n = 0
    for base, dirs, files in os.walk(root):
        depth = base.count("/") - 2
        print("  " * depth + os.path.basename(base) + "/")
        for f in files[:2]:
            print("  " * (depth + 1) + f)
        if len(files) > 2:
            print("  " * (depth + 1) + "... (%d files)" % len(files))
        n += 1
        if n > limit:
            print("  ...truncated")
            return

def find_containing(*markers, root="/kaggle/input"):
    """First directory holding any of `markers` as a child."""
    for base, dirs, files in os.walk(root):
        for m in markers:
            if m in dirs or m in files:
                return base
    return None

CODE = find_containing("pointdenoise")
# The data archive unpacks to examples/ + PUNet/ + PCNet/. Match any of them so
# a renamed or partial upload still resolves.
DATA = find_containing("examples", "PUNet", "PCNet")

print("code:", CODE)
print("data:", DATA)

if not CODE or not DATA:
    print()
    print("=== what is actually mounted at /kaggle/input ===")
    show_tree()
    missing = "pointdenoise-code" if not CODE else "pointdenoise-data"
    raise SystemExit(
        "\nMissing the " + missing + " dataset. Use 'Add Input' in the right-hand "
        "panel and attach BOTH pointdenoise-code and pointdenoise-data (plus the "
        "checkpoint dataset for the benchmark notebook). The tree above shows "
        "what is mounted right now."
    )

sys.path.insert(0, CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "trimesh", "rtree"], check=False)

import torch
print()
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NO GPU - turn it on in Settings > Accelerator")


In [ ]:
CKPT = None
for base, _, files in os.walk("/kaggle/input"):
    if "best.pt" in files:
        CKPT = os.path.join(base, "best.pt"); break
assert CKPT, "best.pt not found - add the dataset holding your checkpoint"
print("checkpoint:", CKPT)

from pointdenoise.engine import load_model
model, ck = load_model(CKPT)
print(f"epoch {ck.get('epoch')}, best loss {ck.get('best'):.6f}, kwargs {ck.get('model_kwargs')}")


## Calibrate first

Scores the bilateral filter, whose numbers are in the published table. Each
metric is checked separately: run 1 passed on CD at 0.84x while P2M sat at
0.17x, so P2M measures something different from what the papers report and
must not be quoted until that is resolved.


In [ ]:
from pointdenoise.benchmark import calibrate, load_released_set

case = load_released_set(DATA, "PUNet", "sparse", 0.01)
r = calibrate(case)
for m in ("cd", "p2m"):
    print(f"  {m.upper():<4} ours {r['measured_'+m]:7.3f}  published {r['expected_'+m]:6.2f}"
          f"  ratio {r[m+'_ratio']:.2f}x  {'PASS' if r[m+'_ok'] else 'FAIL'}")
print(f"\n  quotable: {[m.upper() for m in r['comparable_metrics']]}")
print(f"  not quotable: {[m.upper() for m in r['uncalibrated_metrics']]}")
assert r["cd_ok"], "CD calibration failed - results would not be comparable"


In [ ]:
import numpy as np
from pointdenoise.benchmark import NOISE_LEVELS, load_released_set, run_case
from pointdenoise.data import Shape
from pointdenoise.engine import denoise_cloud
from pointdenoise.metrics import paper_table

def denoiser(points):
    shape = Shape(np.asarray(points), noisy=np.asarray(points))
    return denoise_cloud(model, shape, points_per_patch=256, batch_size=128, iters=1)

# Both halves of the published table. PC-Net is 10 shapes rather than 20, so it
# adds roughly half again on top of the PU-Net pass.
results = {}
for dataset in ("PUNet", "PCNet"):
    scores, baseline = {}, {}
    for resolution in ("sparse", "dense"):
        for noise in NOISE_LEVELS:
            try:
                case = load_released_set(DATA, dataset, resolution, noise)
            except (FileNotFoundError, RuntimeError) as e:
                print(f"skip {dataset}/{resolution}/{noise:.0%}: {e}")
                continue
            _, ours = run_case(case, denoiser, with_p2m=True)
            _, none = run_case(case, lambda p: p, with_p2m=True)
            scores[(resolution, noise)] = ours
            baseline[(resolution, noise)] = none
            gain = (none["cd"] - ours["cd"]) / none["cd"] * 100
            print(f"{dataset}/{resolution}/{noise:.0%}  ours CD {ours['cd']:7.4f}  "
                  f"noisy {none['cd']:7.4f}  {gain:+5.1f}%", flush=True)
    if scores:
        results[dataset] = (scores, baseline)


In [ ]:
CAVEAT = (
    "CD is calibrated: this harness reproduces the published Bilateral CD to "
    "0.84x\non the same shapes, so the CD columns are comparable.\n\n"
    "P2M is NOT calibrated - 0.17x the published value for the same algorithm - "
    "so\nthe P2M columns are shown because the layout calls for them and are not "
    "a claim.\n"
)

out = []
for dataset, (scores, baseline) in results.items():
    published = None if dataset == "PUNet" else {}   # no PC-Net reference table stored
    table = paper_table(scores, our_name="Ours", dataset=dataset, published=published)
    print(table)
    print()
    out.append(table)
    out.append("")
    out.append(f"{dataset} noisy-input baseline (CD x1e-4)")
    for k, v in baseline.items():
        o = scores[k]["cd"]
        out.append(f"  {k[0]}/{k[1]:.0%}  ours {o:7.4f}  noisy {v['cd']:7.4f}  "
                   f"{(v['cd'] - o) / v['cd'] * 100:+5.1f}%")
    out.append("")

with open("/kaggle/working/benchmark.txt", "w") as f:
    f.write("\n".join(out) + "\n\n" + CAVEAT)
print("saved /kaggle/working/benchmark.txt")
